# Sanity Check - Step 00: Load Data Per Person

Überprüft:
- Daten korrekt von BIDS geladen
- Personen (P1 und P2) korrekt aufgeteilt
- Kanäle korrekt zugeordnet
- Datenqualität und Größen

In [1]:
import sys
import mne
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent / 'eeg_pipeline'))
import config

print("Setup erfolgreich")

Setup erfolgreich


## 1. Originaldaten laden

In [ ]:
# Originaldaten von BIDS laden
from mne_bids import BIDSPath, read_raw_bids

subject_id = config.SUBJECTS[0]
bids_path = BIDSPath(
    subject=subject_id,
    task="RPS",
    datatype='eeg',
    suffix='eeg',
    root=config.BIDS_ROOT
)

raw_original = read_raw_bids(bids_path, verbose=False)

print(f"\n=== ORIGINALDATEN ===\n")
print(f"Anzahl Kanäle: {len(raw_original.ch_names)}")
print(f"Sampling Rate: {raw_original.info['sfreq']} Hz")
print(f"Dauer: {raw_original.times[-1]:.2f} Sekunden")

# Berechne Datengröße statt zu laden
n_samples = raw_original.n_times
n_channels = len(raw_original.ch_names)
estimated_size_mb = (n_channels * n_samples * 8) / 1e6  # 8 bytes per float64
print(f"Geschätzte Datengröße: {estimated_size_mb:.2f} MB")

print(f"\nAlle Kanäle:")
for i, ch in enumerate(raw_original.ch_names[:20]):
    print(f"  {i+1}. {ch} - Typ: {raw_original.get_channel_types()[i]}")
if len(raw_original.ch_names) > 20:
    print(f"  ... und {len(raw_original.ch_names) - 20} weitere")

C:\Users\bk57s\AppData\Local\Temp\ipykernel_16672\70296999.py:13: RuntimeWarning: Did not find any channels.tsv associated with sub-01_task-RPS.

The search_str was "c:\Users\bk57s\Visual Studio Code\EEG_Bala Sharks\MNE-sample-data\ds006761\sub-01\**\eeg\sub-01*channels.tsv"
  raw_original = read_raw_bids(bids_path, verbose=False)
C:\Users\bk57s\AppData\Local\Temp\ipykernel_16672\70296999.py:13: RuntimeWarning: Unable to map the following column(s) to to MNE:
date: 9/11/2021
player1_age: 25
player1_gender: F
player1_handedness: R
player1_pre_processing_channels_fixed: 
player2_age: 35
player2_gender: M
player2_handedness: R
player2_pre_processing_channels_fixed: T8
  raw_original = read_raw_bids(bids_path, verbose=False)



=== ORIGINALDATEN ===

Anzahl Kanäle: 143
Sampling Rate: 2048.0 Hz
Dauer: 3669.00 Sekunden


MemoryError: Unable to allocate 8.01 GiB for an array with shape (143, 7514112) and data type float64

## 2. Nach Step 00 verarbeitete Daten laden

In [ ]:
# Step 00 Outputs laden (without preloading to save memory)
p1_path = config.OUTPUT_DIR / f"sub-{subject_id}_P1_raw.fif"
p2_path = config.OUTPUT_DIR / f"sub-{subject_id}_P2_raw.fif"

raw_p1 = mne.io.read_raw_fif(str(p1_path), preload=False)
raw_p2 = mne.io.read_raw_fif(str(p2_path), preload=False)

print(f"\n=== NACH STEP 00 ===\n")

# Calculate estimated sizes instead of loading
def estimate_size_mb(raw):
    n_samples = raw.n_times
    n_channels = len(raw.ch_names)
    return (n_channels * n_samples * 8) / 1e6

print(f"Person 1:")
print(f"  Anzahl Kanäle: {len(raw_p1.ch_names)}")
print(f"  Sampling Rate: {raw_p1.info['sfreq']} Hz")
print(f"  Dauer: {raw_p1.times[-1]:.2f} Sekunden")
print(f"  Geschätzte Datengröße: {estimate_size_mb(raw_p1):.2f} MB")

print(f"\nPerson 2:")
print(f"  Anzahl Kanäle: {len(raw_p2.ch_names)}")
print(f"  Sampling Rate: {raw_p2.info['sfreq']} Hz")
print(f"  Dauer: {raw_p2.times[-1]:.2f} Sekunden")
print(f"  Geschätzte Datengröße: {estimate_size_mb(raw_p2):.2f} MB")

## 3. Vergleich Originaldata vs. Step 00

In [ ]:
# Load mit preload=False um Speicher zu sparen
raw_p1_loaded = mne.io.read_raw_fif(str(p1_path), preload=False)
raw_p2_loaded = mne.io.read_raw_fif(str(p2_path), preload=False)

print(f"\n=== KANÄLE VERGLEICH ===\n")

# Person 1 Kanäle identifizieren
p1_ch_types = {}
for ch_name, ch_type in zip(raw_p1_loaded.ch_names, raw_p1_loaded.get_channel_types()):
    if ch_type not in p1_ch_types:
        p1_ch_types[ch_type] = []
    p1_ch_types[ch_type].append(ch_name)

p2_ch_types = {}
for ch_name, ch_type in zip(raw_p2_loaded.ch_names, raw_p2_loaded.get_channel_types()):
    if ch_type not in p2_ch_types:
        p2_ch_types[ch_type] = []
    p2_ch_types[ch_type].append(ch_name)

print(f"Person 1 - Kanäle nach Typ:")
for ch_type, channels in sorted(p1_ch_types.items()):
    print(f"  {ch_type}: {len(channels)} Kanäle")
    if ch_type == 'eeg':
        print(f"    Kanäle: {channels[:5]}{'...' if len(channels) > 5 else ''}")

print(f"\nPerson 2 - Kanäle nach Typ:")
for ch_type, channels in sorted(p2_ch_types.items()):
    print(f"  {ch_type}: {len(channels)} Kanäle")
    if ch_type == 'eeg':
        print(f"    Kanäle: {channels[:5]}{'...' if len(channels) > 5 else ''}")

## 4. Datenintegrität prüfen

In [ ]:
# Prüfe auf NaN und Inf Werte - lade nur EEG Kanäle für erste 60 Sekunden um Speicher zu sparen
eeg_picks_p1 = mne.pick_types(raw_p1.info, eeg=True)
eeg_picks_p2 = mne.pick_types(raw_p2.info, eeg=True)

# Lade nur Subset für Integrität-Check
tmax = min(60, raw_p1.times[-1])
data_p1_eeg = raw_p1.get_data(picks=eeg_picks_p1, tmax=tmax)
data_p2_eeg = raw_p2.get_data(picks=eeg_picks_p2, tmax=tmax)

print(f"\n=== DATENINTEGRITÄT (erste {tmax:.0f}s) ===\n")

print(f"Person 1 (EEG Kanäle)::")
print(f"  NaN Werte: {np.isnan(data_p1_eeg).sum()}")
print(f"  Inf Werte: {np.isinf(data_p1_eeg).sum()}")
print(f"  Min Wert: {np.nanmin(data_p1_eeg):.6f}")
print(f"  Max Wert: {np.nanmax(data_p1_eeg):.6f}")
print(f"  Mean: {np.nanmean(data_p1_eeg):.6f}")
print(f"  Std: {np.nanstd(data_p1_eeg):.6f}")

print(f"\nPerson 2 (EEG Kanäle)::")
print(f"  NaN Werte: {np.isnan(data_p2_eeg).sum()}")
print(f"  Inf Werte: {np.isinf(data_p2_eeg).sum()}")
print(f"  Min Wert: {np.nanmin(data_p2_eeg):.6f}")
print(f"  Max Wert: {np.nanmax(data_p2_eeg):.6f}")
print(f"  Mean: {np.nanmean(data_p2_eeg):.6f}")
print(f"  Std: {np.nanstd(data_p2_eeg):.6f}")

# Speichere für später
data_p1 = data_p1_eeg
data_p2 = data_p2_eeg

## 5. Visualisierung der Raw-Daten

In [ ]:
# Wähle nur EEG Kanäle und begrenzte Zeit für Visualisierung
eeg_p1 = raw_p1.copy().pick_types(eeg=True)
eeg_p2 = raw_p2.copy().pick_types(eeg=True)

# PSD (Power Spectral Density) plotten - begrenzt auf erste 300 Sekunden
fig = plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
eeg_p1.plot_psd(fmax=50, tmax=300, ax=plt.gca(), show=False)
plt.title('Person 1 - PSD (erste 300s)')

plt.subplot(1, 2, 2)
eeg_p2.plot_psd(fmax=50, tmax=300, ax=plt.gca(), show=False)
plt.title('Person 2 - PSD (erste 300s)')

plt.tight_layout()
plt.show()

## 6. Ereignisse prüfen

In [ ]:
# Events identifizieren (ohne vollständiges Laden)
events_p1, event_id_p1 = mne.events_from_annotations(raw_p1)
events_p2, event_id_p2 = mne.events_from_annotations(raw_p2)

print(f"\n=== EREIGNISSE ===\n")
print(f"Person 1:")
print(f"  Anzahl Ereignisse: {len(events_p1)}")
print(f"  Ereignis-IDs: {event_id_p1}")
print(f"\nPerson 2:")
print(f"  Anzahl Ereignisse: {len(events_p2)}")
print(f"  Ereignis-IDs: {event_id_p2}")

## 7. Zusammenfassung & Validierung

In [ ]:
print(f"\n=== SANITY CHECK ZUSAMMENFASSUNG ===\n")

# Prüfe kritische Bedingungen
checks = []

# 1. Zeitdauer sollte gleich sein
checks.append(("Zeitdauer gleich", 
                abs(raw_p1_loaded.times[-1] - raw_p2_loaded.times[-1]) < 0.01))

# 2. Sampling Rate sollte gleich sein
checks.append(("Sampling Rate gleich", 
                raw_p1_loaded.info['sfreq'] == raw_p2_loaded.info['sfreq']))

# 3. EEG Kanäle sollten vorhanden sein
checks.append(("EEG Kanäle (P1)", 
                len(mne.pick_types(raw_p1_loaded.info, eeg=True)) > 0))

checks.append(("EEG Kanäle (P2)", 
                len(mne.pick_types(raw_p2_loaded.info, eeg=True)) > 0))

# 4. Keine NaN Werte
checks.append(("Keine NaN Werte (P1)", 
                np.isnan(data_p1).sum() == 0))

checks.append(("Keine NaN Werte (P2)", 
                np.isnan(data_p2).sum() == 0))

# 5. Ereignisse sollten vorhanden sein
checks.append(("Ereignisse vorhanden (P1)", 
                len(events_p1) > 0))

checks.append(("Ereignisse vorhanden (P2)", 
                len(events_p2) > 0))

for check_name, result in checks:
    status = "✓ PASS" if result else "✗ FAIL"
    print(f"{status}: {check_name}")

all_pass = all(result for _, result in checks)
print(f"\n{'='*50}")
if all_pass:
    print("✓ ALLE CHECKS BESTANDEN")
else:
    print("✗ EINIGE CHECKS FEHLGESCHLAGEN")
print(f"{'='*50}")